# Exploratory analysis
Loads raw and scored results. Outputs are cleared in git; run after `scripts/analyse_results.py`. **Mock data (`results_smoke`) says nothing about real models.**

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from reliability_lab.pipeline.result_collector import read_raw, load_scored
from reliability_lab.evaluation.metrics import compute_metrics
RESULTS = "results"   # or "results_smoke" (MOCK data: pipeline testing only)

In [ ]:
raw = read_raw(RESULTS); scored = load_scored(RESULTS)
print(len(raw), 'raw;', len(scored), 'scored')
raw.groupby(['model_key','experiment']).size().unstack(fill_value=0)

## Data quality: errors, truncation, missing answer lines

In [ ]:
raw['error'].notna().groupby(raw['model_key']).mean().rename('error_rate')

In [ ]:
if 'finish_reason' in raw: display(raw.groupby(['model_key','finish_reason']).size().unstack(fill_value=0))

## Metrics with 95% intervals

In [ ]:
m = compute_metrics(scored, {'bootstrap_resamples': 1000})
m[m.subset=='all'][['experiment','model_key','metric','value','ci_low','ci_high','n']].round(3)

## Pass-rate heatmap (experiment x model)

In [ ]:
pr = scored.groupby(['experiment','model_key'])['passed'].mean().unstack()
fig, ax = plt.subplots(figsize=(6,3.5)); im = ax.imshow(pr.values, vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(pr.shape[1]), pr.columns); ax.set_yticks(range(pr.shape[0]), pr.index)
for i in range(pr.shape[0]):
    for j in range(pr.shape[1]): ax.text(j, i, f'{pr.values[i,j]:.2f}', ha='center', va='center')
plt.colorbar(im); plt.tight_layout()

## Context sensitivity by condition

In [ ]:
cs = scored[scored.experiment=='context_sensitivity']
cs.groupby(['model_key','condition'])['correct'].mean().unstack().round(2)

## Response length by model (stylistic variation)

In [ ]:
scored.assign(n_words=scored.response.str.split().str.len()).groupby(['experiment','model_key']).n_words.describe().round(1)